# 問い合わせを担当部署に振り分け、FAQ から返信の下書きを作る

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mamorlis/nlpcookbook/blob/main/notebooks/22_dialogue/ticket_routing.ipynb)

NLP クックブック 22-03 ／ 対話システム

通販サイトに届く問い合わせのメールを、担当の部署 (配送、返品・交換、支払い、アカウント、商品) に自動で振り分けたい。あわせて FAQ を検索し、オペレーターが確認して送るだけの返信の下書きを作りたい。自信のない振り分けや、FAQ に答えのない問い合わせは人に回したい。

In [1]:
import numpy as np
import pandas as pd
from sentence_transformers import SentenceTransformer

# 架空の通販サイトの FAQ と問い合わせ (このレシピのために作った例)
FAQ = {
    "F01": ("配送", ["送料はいくらですか", "送料無料になる条件は", "配送料を教えてください"],
            "送料は全国一律 550 円です。5,000 円以上のご注文で無料になります。"),
    "F02": ("配送", ["届くまで何日かかりますか", "注文した商品がまだ届かない", "配送状況を知りたい", "いつ発送されますか"],
            "通常、ご注文から 2〜4 日でお届けします。発送後にお送りするメールの追跡番号で配送状況を確認できます。"),
    "F03": ("配送", ["届け先の住所を変更したい", "配送先を間違えて入力した"],
            "発送前であれば、マイページの注文履歴からお届け先を変更できます。発送後の変更は配送業者へ直接ご連絡ください。"),
    "F04": ("配送", ["配達日を指定できますか", "配達の時間帯を指定したい"],
            "ご注文手続きの画面で、お届け希望日 (ご注文の 3 日後から 14 日後まで) と時間帯を指定できます。"),
    "F05": ("返品・交換", ["返品したい", "返品の期限はいつまでですか", "返品の手続きの方法"],
            "商品到着から 14 日以内であれば返品できます。マイページの注文履歴からお手続きください。"),
    "F06": ("返品・交換", ["届いた商品が壊れていた", "注文と違う商品が届いた", "不良品だった"],
            "届いた商品に破損や誤りがあった場合は、お手数ですが商品の写真を添えてご連絡ください。送料は当店負担で交換いたします。"),
    "F07": ("返品・交換", ["サイズを交換したい", "大きいサイズに替えてほしい"],
            "サイズ交換は商品到着から 14 日以内、未使用の商品に限り承ります。マイページの注文履歴から「交換」を選んでください。"),
    "F08": ("支払い", ["支払い方法は何がありますか", "クレジットカードは使えますか", "コンビニ払いはできますか"],
            "クレジットカード、コンビニ払い、銀行振込がご利用いただけます。"),
    "F09": ("支払い", ["領収書がほしい", "領収書の宛名を指定したい"],
            "領収書はマイページの注文履歴から PDF でダウンロードできます。宛名は発行時に入力できます。"),
    "F10": ("支払い", ["返品した商品の返金はいつですか", "返金されていない"],
            "返品された商品の到着を確認してから、5 営業日以内に返金いたします。クレジットカードの場合、反映は締め日によって翌月以降になることがあります。"),
    "F11": ("アカウント", ["パスワードを忘れました", "ログインできない", "パスワードを変更したい"],
            "ログイン画面の「パスワードをお忘れの方」から再設定できます。"),
    "F12": ("アカウント", ["登録しているメールアドレスを変更したい"],
            "マイページの「会員情報」から、登録メールアドレスを変更できます。"),
    "F13": ("アカウント", ["退会したい", "退会するとポイントはどうなりますか"],
            "マイページの「会員情報」の一番下にある「退会」から手続きできます。保有ポイントは退会と同時に失効します。"),
    "F14": ("商品", ["在庫切れの商品は再入荷しますか", "入荷したら知らせてほしい"],
            "在庫切れの商品は、商品ページの「再入荷のお知らせ」に登録すると、入荷時にメールでお知らせします。"),
    "F15": ("商品", ["ギフト包装はできますか", "プレゼント用のラッピングの料金"],
            "ギフト包装 (1 点 300 円) は、ご注文手続きの画面で選べます。"),
}
# 部署の説明 (ゼロショット分類に使う)
DEPTS = {
    "配送": "商品の配送、お届け日、送料、配送先の変更、荷物が届かないことに関する問い合わせ",
    "返品・交換": "商品の返品、交換、破損や不良品、注文と違う商品が届いたことに関する問い合わせ",
    "支払い": "代金の支払い方法、請求、返金、領収書、クーポンに関する問い合わせ",
    "アカウント": "会員登録、ログイン、パスワード、メールアドレス、退会、ポイントに関する問い合わせ",
    "商品": "商品の仕様、サイズ、素材、在庫、再入荷、ギフト包装に関する問い合わせ",
}
# 問い合わせ 100 件: 部署|答えが載っている FAQ (なければ -)|本文
TICKETS = """
配送|F02|先週の月曜日に注文したのですが、まだ届きません。いつ頃届きますか。
配送|F02|注文番号 A1203 の荷物は今どこにありますか？追跡の方法を教えてください。
配送|F01|送料はいくらかかりますか。北海道に住んでいます。
配送|F01|あと少しで送料無料になるのですが、何円以上なら無料ですか？
配送|F03|引っ越すことになったので、届け先を新しい住所に変えたいです。まだ発送されていないと思います。
配送|F03|お届け先の住所の番地を間違えて入力してしまいました。直せますか。
配送|F04|来週の土曜日の午前中に届けてもらうことはできますか。
配送|F04|配達の時間帯を夜にしてほしいのですが、どこで指定すればいいですか。
配送|-|配達員の方が不在票を入れずに持ち帰ってしまったようです。
配送|-|荷物が「配達完了」になっているのに、家に届いていません。
配送|F02|お世話になっております。13 日に注文した件ですが、発送のメールが来ません。いつ発送されますか。
配送|-|置き配を指定したのに、玄関前ではなく宅配ボックスに入れられていました。
配送|F02|急ぎで必要なのですが、注文してから何日で届きますか。
配送|-|海外 (台湾) への発送はしていますか。
配送|F03|発送済みのメールが来ましたが、受け取り場所を勤務先に変えたいです。
配送|-|届いた箱がつぶれていました。中身は無事でしたが、梱包をもう少し丁寧にしてほしいです。
配送|F01|離島への配送でも送料は同じですか。
配送|F04|誕生日に間に合うように、日付を指定して送れますか。
配送|-|コンビニで受け取ることはできますか。
配送|F02|2 つ注文したうち 1 つしか届いていません。残りはいつ届きますか。
返品・交換|F05|サイズが合わなかったので返品したいです。
返品・交換|F05|届いて 1 週間ですが、まだ返品できますか。
返品・交換|F06|届いたマグカップが割れていました。
返品・交換|F06|注文した色と違う商品が届きました。青を頼んだのに赤でした。
返品・交換|F07|M サイズを買いましたが小さかったので、L に交換してほしいです。
返品・交換|F07|靴のサイズ交換をお願いできますか。一度も履いていません。
返品・交換|F05|イメージと違ったので返品したいのですが、手続きの方法がわかりません。
返品・交換|-|返品した商品の受け取り確認の連絡がまだありません。
返品・交換|F06|電源を入れても動きません。初期不良だと思います。
返品・交換|-|開封済みのものでも返品できますか。
返品・交換|F05|商品が届いてから何日以内なら返品を受け付けてもらえますか。
返品・交換|-|セール品は返品できますか。
返品・交換|F06|部品が 1 つ足りませんでした。
返品・交換|-|色違いのものに交換することはできますか。
返品・交換|-|返品するときの送料はどちらの負担ですか。
返品・交換|F06|中身が別の商品と入れ違っているようです。注文番号は B2201 です。
返品・交換|-|プレゼントでもらったものを返品したいのですが可能ですか。
返品・交換|F06|洗濯したら一回で破れてしまいました。不良品ではないでしょうか。
返品・交換|-|注文した覚えのない商品が同梱されていたので返送したいです。
返品・交換|F07|子どもの服のサイズを 1 つ大きいものに替えてもらえますか。
支払い|F08|コンビニで支払えますか。
支払い|F08|使えるクレジットカードの種類を教えてください。
支払い|-|同じ注文で 2 回請求されています。確認してください。
支払い|F09|会社の経費で落としたいので領収書がほしいです。
支払い|F09|領収書の宛名を会社名にできますか。
支払い|F10|返品した分の返金はいつになりますか。
支払い|F10|返品から 2 週間たちますが、まだカードに返金されていません。
支払い|-|銀行振込の振込先を教えてください。
支払い|-|後払いはできますか。
支払い|-|クーポンコードを入れたのに割引されていません。
支払い|-|請求金額が商品ページの表示と違います。
支払い|-|インボイス制度に対応した領収書は発行できますか。
支払い|-|コンビニ払いの期限を過ぎてしまいました。注文はどうなりますか。
支払い|-|PayPay は使えますか。
支払い|-|ポイントは支払いに使えますか。
支払い|-|キャンセルした注文の代金はいつ戻ってきますか。
支払い|-|カード決済がエラーになって注文できません。
支払い|F09|領収書を再発行してほしいです。
支払い|-|分割払いにしたいのですが、注文後に変更できますか。
支払い|-|振り込んだのに入金確認のメールが来ません。
アカウント|F11|パスワードを忘れてログインできません。
アカウント|F11|ログインしようとすると「パスワードが違います」と出ます。
アカウント|F12|メールアドレスを変えたので、登録しているアドレスも変更したいです。
アカウント|F13|退会したいです。
アカウント|F13|退会を考えていますが、貯まっているポイントは残りますか。
アカウント|-|メールマガジンの配信を止めてほしいです。
アカウント|-|会員登録の確認メールが届きません。
アカウント|F12|携帯のメールアドレスから Gmail に変更したいのですが、どこで変えられますか。
アカウント|-|アカウントが乗っ取られたかもしれません。身に覚えのない注文があります。
アカウント|-|パスワード再設定のメールのリンクを開いたら期限切れでした。
アカウント|-|登録している電話番号を変えたいです。
アカウント|-|アカウントを 2 つ作ってしまったので 1 つにまとめたいです。
アカウント|F13|もう利用しないので、会員情報を削除してください。
アカウント|-|ポイントの有効期限を確認したいです。
アカウント|F11|パスワードを変更したいです。
アカウント|-|2 段階認証を設定できますか。
アカウント|-|ログインするたびにログアウトされてしまいます。
アカウント|F12|登録したメールアドレスを間違えていました。正しいものに直したいです。
アカウント|-|会員ランクの条件を教えてください。
アカウント|-|ログイン ID を忘れました。
商品|F14|欲しい商品が在庫切れになっています。再入荷はいつですか。
商品|F14|売り切れのスニーカーが入荷したら知らせてもらえますか。
商品|F15|プレゼント用に包装してもらえますか。
商品|F15|ギフトラッピングの料金はいくらですか。
商品|-|このジャケットは洗濯機で洗えますか。
商品|-|身長 160cm だと S と M のどちらがいいですか。
商品|-|この電気ケトルの容量は何リットルですか。
商品|-|商品ページの写真と実物の色は同じですか。
商品|-|この鍋は IH に対応していますか。
商品|F14|品切れの色はもう入荷しないのでしょうか。
商品|-|まとめて 50 個注文したいのですが、法人割引はありますか。
商品|-|このバッグに A4 の書類は入りますか。
商品|-|保証期間は何年ですか。
商品|-|のし紙を付けてもらうことはできますか。
商品|-|この商品の原産国はどこですか。
商品|-|子ども用のサイズはありますか。
商品|-|入荷待ちの商品を予約注文できますか。
商品|-|アレルギーがあるのですが、この石けんの成分を教えてください。
商品|-|替えのフィルターは売っていますか。
商品|-|取扱説明書をなくしてしまいました。
"""
df = pd.DataFrame([line.split("|") for line in TICKETS.strip().split("\n")], columns=["dept", "faq", "text"])
print(df.groupby("dept").agg(件数=("text", "size"), FAQ_あり=("faq", lambda s: (s != "-").sum())))

model = SentenceTransformer("intfloat/multilingual-e5-small")   # MIT ライセンス
X = model.encode(["query: " + t for t in df["text"]], normalize_embeddings=True)

       件数  FAQ_あり
dept             
アカウント  20       9
商品     20       5
支払い    20       7
返品・交換  20      13
配送     20      14


In [2]:
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import classification_report, confusion_matrix
from sklearn.model_selection import StratifiedKFold, cross_val_predict

def accuracy_ci(correct, n_boot=2000, seed=0):
    """正解率と、ブートストラップによる 95% 信頼区間"""
    rng = np.random.default_rng(seed)
    boot = [correct[rng.integers(0, len(correct), len(correct))].mean() for _ in range(n_boot)]
    return f"{correct.mean():.3f} [{np.percentile(boot, 2.5):.3f}, {np.percentile(boot, 97.5):.3f}]"

y = df["dept"].to_numpy()
names = list(DEPTS)

# 1. ゼロショット: 部署の説明文との類似度 (学習データなし)
dept_emb = model.encode(["query: " + d for d in DEPTS.values()], normalize_embeddings=True)
pred_zs = np.array(names)[(X @ dept_emb.T).argmax(1)]
print("ゼロショット (説明文との類似度):", accuracy_ci(pred_zs == y))

# 2. 埋め込み + ロジスティック回帰: 5 分割交差検証で、各問い合わせを「学習に使っていないモデル」で予測する
cv = StratifiedKFold(5, shuffle=True, random_state=0)
clf = LogisticRegression(C=1000, max_iter=5000)                # 例が少ないので正則化を弱くする (C が小さいと確率が平らになる)
proba = cross_val_predict(clf, X, y, cv=cv, method="predict_proba")
classes = np.array(sorted(names))                       # predict_proba の列は classes_ (文字列の昇順)
pred_lr = classes[proba.argmax(1)]
print("埋め込み + ロジスティック回帰 (交差検証):", accuracy_ci(pred_lr == y))
print(classification_report(y, pred_lr, digits=3, zero_division=0))
print(pd.DataFrame(confusion_matrix(y, pred_lr, labels=names), index=names, columns=names))
for t, p, gold in zip(df["text"], pred_lr, y):
    if p != gold:
        print(f"  誤り: 正解 {gold} → 予測 {p}  {t}")

ゼロショット (説明文との類似度): 0.750 [0.660, 0.840]
埋め込み + ロジスティック回帰 (交差検証): 0.890 [0.830, 0.950]
              precision    recall  f1-score   support

       アカウント      1.000     0.950     0.974        20
          商品      0.944     0.850     0.895        20
         支払い      0.895     0.850     0.872        20
       返品・交換      0.760     0.950     0.844        20
          配送      0.895     0.850     0.872        20

    accuracy                          0.890       100
   macro avg      0.899     0.890     0.891       100
weighted avg      0.899     0.890     0.891       100

       配送  返品・交換  支払い  アカウント  商品
配送     17      2    1      0   0
返品・交換   0     19    0      0   1
支払い     1      2   17      0   0
アカウント   1      0    0     19   0
商品      0      2    1      0  17
  誤り: 正解 配送 → 予測 返品・交換  お届け先の住所の番地を間違えて入力してしまいました。直せますか。
  誤り: 正解 配送 → 予測 返品・交換  届いた箱がつぶれていました。中身は無事でしたが、梱包をもう少し丁寧にしてほしいです。
  誤り: 正解 配送 → 予測 支払い  コンビニで受け取ることはできますか。
  誤り: 正解 返品・交換 → 予測 商品  届いたマグカップが割れていました。
  誤り: 正解 支払い → 予測 返品

In [3]:
# 自信のない振り分けは人に回す: 最大確率がしきい値未満なら「要確認」
conf = proba.max(1)
rows = []
for th in [0.0, 0.5, 0.6, 0.7, 0.8, 0.9]:
    auto = conf >= th
    rows.append({"しきい値": th, "自動で振り分け": f"{auto.mean():.0%}",
                 "自動分の正解率": round((pred_lr[auto] == y[auto]).mean(), 3) if auto.any() else None,
                 "誤って自動振り分け (件)": int((auto & (pred_lr != y)).sum())})
print(pd.DataFrame(rows).to_string(index=False))

 しきい値 自動で振り分け  自動分の正解率  誤って自動振り分け (件)
  0.0    100%    0.890             11
  0.5     92%    0.924              7
  0.6     88%    0.932              6
  0.7     76%    0.974              2
  0.8     65%    1.000              0
  0.9     48%    1.000              0


In [4]:
import re

faq_ids = [(fid, q) for fid, (_, qs, _) in FAQ.items() for q in qs]       # 1 つの FAQ に複数の言い回しを登録しておく
faq_emb = model.encode(["query: " + q for _, q in faq_ids], normalize_embeddings=True)

FAQ_OF = [fid for fid, _ in faq_ids]

def search_faq(texts):
    """問い合わせを文に分け、各 FAQ について「どれかの文とどれかの言い回し」の最大類似度を求め、高い順に返す"""
    sents = [(i, s) for i, t in enumerate(texts) for s in re.split(r"(?<=[。？！?!])\s*", t) if s.strip()]
    owner = np.array([i for i, _ in sents])
    sims = model.encode(["query: " + s for _, s in sents], normalize_embeddings=True, batch_size=64) @ faq_emb.T
    results = []
    for i in range(len(texts)):
        best = {}
        for fid, s in zip(FAQ_OF, sims[owner == i].max(0)):   # 言い回しごとの「どれかの文との最大類似度」
            best[fid] = max(best.get(fid, 0), float(s))         # FAQ ごとに、言い回しの中の最大をとる
        results.append(sorted(best.items(), key=lambda x: -x[1]))
    return results

hits = search_faq(df["text"].tolist())
top_id = np.array([h[0][0] for h in hits])
top_score = np.array([h[0][1] for h in hits])
has_faq = (df["faq"] != "-").to_numpy()
print("FAQ に答えがある問い合わせ:", has_faq.sum(), "件 / 1 位の FAQ が正しい:", (top_id[has_faq] == df["faq"][has_faq]).sum(), "件")

# FAQ の 1 位の類似度がしきい値以上のときだけ下書きを作る。しきい値を変えて、下書きの量と正しさを見る
rows = []
for th in [0.86, 0.88, 0.90, 0.91, 0.92, 0.94]:
    draft = top_score >= th
    ok = draft & (top_id == df["faq"].to_numpy())
    rows.append({"しきい値": th, "下書きを作った": int(draft.sum()), "正しい FAQ に基づく": int(ok.sum()),
                 "答えのない問い合わせに下書き": int((draft & ~has_faq).sum()),
                 "答えのある問い合わせを拾えた": f"{ok[has_faq].mean():.0%}"})
print(pd.DataFrame(rows).to_string(index=False))
for t, fid, sc, gold in zip(df["text"], top_id, top_score, df["faq"]):
    if sc >= 0.91 and fid != gold:              # しきい値 0.91 でも、答えにならない FAQ で下書きを作ってしまうもの
        print(f"  {sc:.3f} {fid} (正解 {gold})  {t}")

FAQ に答えがある問い合わせ: 48 件 / 1 位の FAQ が正しい: 44 件
 しきい値  下書きを作った  正しい FAQ に基づく  答えのない問い合わせに下書き 答えのある問い合わせを拾えた
 0.86       91            44              43            92%
 0.88       82            43              36            90%
 0.90       65            40              24            83%
 0.91       50            34              15            71%
 0.92       38            30               8            62%
 0.94       22            21               1            44%
  0.937 F06 (正解 -)  届いた箱がつぶれていました。中身は無事でしたが、梱包をもう少し丁寧にしてほしいです。
  0.911 F06 (正解 F01)  離島への配送でも送料は同じですか。
  0.930 F08 (正解 -)  コンビニで受け取ることはできますか。
  0.920 F02 (正解 -)  返品した商品の受け取り確認の連絡がまだありません。
  0.913 F05 (正解 -)  開封済みのものでも返品できますか。
  0.923 F05 (正解 -)  セール品は返品できますか。
  0.917 F10 (正解 -)  返品するときの送料はどちらの負担ですか。
  0.938 F05 (正解 -)  プレゼントでもらったものを返品したいのですが可能ですか。
  0.911 F06 (正解 -)  同じ注文で 2 回請求されています。確認してください。
  0.916 F01 (正解 -)  コンビニ払いの期限を過ぎてしまいました。注文はどうなりますか。
  0.916 F02 (正解 -)  カード決済がエラーになって注文できません。
  0.914 F06 (正解 -)  アカウントが乗っ取られたかもしれません。身に覚え

In [5]:
# 振り分けと下書きをまとめる。分類器は 100 件すべてで学習し直す
# (しきい値は 80 件で学習した交差検証の確率で決めたので、100 件で学習したモデルでは確率の出方が少し違う)
clf.fit(X, y)
ROUTE_TH, FAQ_TH, REF_TH = 0.7, 0.91, 0.88    # しきい値は上の表を見て決めた値 (同じデータで決めたので、新しいデータで確かめ直す)

def handle(text, max_faq=2):
    p = clf.predict_proba(model.encode(["query: " + text], normalize_embeddings=True))[0]
    dept = str(clf.classes_[p.argmax()])
    route = dept if p.max() >= ROUTE_TH else f"要確認 (候補: {dept})"
    ranked = [(fid, round(s, 3)) for fid, s in search_faq([text])[0][:max_faq]]
    used = [(fid, s) for fid, s in ranked if s >= FAQ_TH]
    refs = [(fid, s) for fid, s in ranked if REF_TH <= s < FAQ_TH]     # 下書きには使わず、担当者に参考として見せる
    draft = None
    if used:          # 根拠になる FAQ がなければ、下書きを作らずに担当者に任せる
        body = "\n".join(FAQ[fid][2] for fid, _ in used)   # 回答部分は FAQ の文をそのまま使い、新しい事実を足さない
        draft = f"お問い合わせいただきありがとうございます。\n{body}\nほかにご不明な点がございましたら、お気軽にお問い合わせください。"
    return route, p.max(), used, refs, draft

for text in ["注文した T シャツが小さかったので、L サイズに交換してください。あと、会社に出すので領収書もほしいです。",
             "3 日前に注文しましたが、まだ発送のお知らせがありません。",
             "退会の方法を教えてください。",
             "届いたお皿が欠けていました。返金してもらえますか。",
             "御社の株主優待について知りたいです。"]:
    route, conf, used, refs, draft = handle(text)
    print(f"問い合わせ: {text}\n  振り分け: {route} (確率 {conf:.2f})  根拠: {used}  参考: {refs}")
    print("  " + (draft.replace("\n", "\n  ") if draft else "(下書きなし: 担当者が回答を作成)") + "\n")

問い合わせ: 注文した T シャツが小さかったので、L サイズに交換してください。あと、会社に出すので領収書もほしいです。
  振り分け: 要確認 (候補: 返品・交換) (確率 0.52)  根拠: [('F09', 0.947)]  参考: [('F07', 0.907)]
  お問い合わせいただきありがとうございます。
  領収書はマイページの注文履歴から PDF でダウンロードできます。宛名は発行時に入力できます。
  ほかにご不明な点がございましたら、お気軽にお問い合わせください。

問い合わせ: 3 日前に注文しましたが、まだ発送のお知らせがありません。
  振り分け: 配送 (確率 0.71)  根拠: []  参考: [('F02', 0.901)]
  (下書きなし: 担当者が回答を作成)

問い合わせ: 退会の方法を教えてください。
  振り分け: アカウント (確率 0.99)  根拠: [('F13', 0.961)]  参考: []
  お問い合わせいただきありがとうございます。
  マイページの「会員情報」の一番下にある「退会」から手続きできます。保有ポイントは退会と同時に失効します。
  ほかにご不明な点がございましたら、お気軽にお問い合わせください。

問い合わせ: 届いたお皿が欠けていました。返金してもらえますか。
  振り分け: 返品・交換 (確率 0.86)  根拠: [('F06', 0.931), ('F10', 0.924)]  参考: []
  お問い合わせいただきありがとうございます。
  届いた商品に破損や誤りがあった場合は、お手数ですが商品の写真を添えてご連絡ください。送料は当店負担で交換いたします。
  返品された商品の到着を確認してから、5 営業日以内に返金いたします。クレジットカードの場合、反映は締め日によって翌月以降になることがあります。
  ほかにご不明な点がございましたら、お気軽にお問い合わせください。

問い合わせ: 御社の株主優待について知りたいです。
  振り分け: 要確認 (候補: 支払い) (確率 0.44)  根拠: []  参考: []
  (下書きなし: 担当者が回答を作成)



- 流れは「部署の分類 → 自信のないものを人に回す → FAQ の検索 → FAQ の文だけで下書き」。分類はこの章の「発話の意図を分類する (少数の例から)」、検索は「よくある質問に自動で答える FAQ ボットを作る」と同じ部品で、問い合わせのメールを扱うために、交差検証による評価、しきい値の決め方、複数の文からなる問い合わせの検索を加えた。FAQ と問い合わせは、このレシピのために作った架空の例である。FAQ に載っていない方法を尋ねるもの (「後払いはできますか」など) は「答えなし」としている。
- 部署の分類の正解率は、部署の説明文との類似度によるゼロショットで 0.750 (95% 信頼区間 0.660〜0.840)、各部署 20 件の例で学習したロジスティック回帰 (5 分割交差検証) で 0.890 (0.830〜0.950) だった。評価は 100 件しかないので区間は広いが、数十件の例を用意できるなら学習するほうがよい。誤りの多くは「割れていた」(返品・交換) を商品に、「返品した分の返金」(支払い) を返品・交換に、のように部署の境目で起きた。こうした問い合わせをどこが受け持つかは、分類器の前に業務の規則として決めておく。「アカウントが乗っ取られた」を配送に、「取扱説明書をなくした」を返品・交換に、のような境目とは言えない誤りもあり、こちらは例を足して直す。
- しきい値は、交差検証で各問い合わせを「学習に使っていないモデル」で予測した確率から決める。学習データで予測すると確率が高く出て、しきい値が甘くなる。この例では最大確率 0.7 以上を自動で振り分けると、76% を自動で処理し、そのうち誤りは 2 件 (正解率 0.974) で、残りの 24% が人の確認に回った。0.8 にすると誤りは 0 件になるが、自動は 65% に減る。分類器は意図の分類のレシピと同じく `C=1000` で正則化を弱くしている。`C` が小さいと確率が全体に平らになり、多くが「要確認」に回る。最後のセルは 100 件すべてで学習し直したモデルにこのしきい値を当てているので、確率の出方は交差検証のときと少し違う (「3 日前に注文しました」は 0.71 で、ぎりぎり自動になった)。確率がどのくらい当てになるかは、評価指標と統計的検定の章の「モデルの確信度が当てになるか調べる (較正、ECE)」で確かめられる。
- 問い合わせは複数の文や用件を含むので、文に分けて検索し、FAQ ごとに最大の類似度をとる。答えが FAQ にある 48 件では、1 位の FAQ が 44 件で正しかった。最初の T シャツの例では、検索の 1 位と 2 位に領収書 (0.947) と交換 (0.907) の FAQ がそれぞれ来る。
- 難しいのは「話題は近いが、答えは FAQ にない」問い合わせを見分けること。類似度のしきい値 0.91 では下書きを 50 件作り、正しい FAQ に基づくのは 34 件で、15 件は答えのない問い合わせ (「開封済みのものでも返品できますか」に返品期限の FAQ を当てるなど) だった。0.94 まで上げると誤りは 1 件になるが、答えのある問い合わせの 44% しか拾えない。RAG の章の「答えが資料にないときに「わかりません」と答えさせる」と同じ問題で、類似度だけでは解決しない。このレシピでは、しきい値の手前の FAQ を下書きに使わず「参考」として担当者に見せている。
- 下書きの回答部分は FAQ の文をそのまま並べるので、FAQ にない事実 (日数や金額) を作ることはない。ただし、答えになっていない下書きはできる。T シャツの例は交換の FAQ がしきい値に届かず、領収書にしか答えていない。お皿の例では、交換の FAQ と並んで「返品された商品」の返金の FAQ が入り、交換と返金のどちらで対応するのかがはっきりしない文面になった。「3 日前に注文しました」は配送に振り分けられたが、配送状況の FAQ (0.901) がしきい値に届かず、下書きは作られなかった (FAQ は参考として表示される)。下書きは担当者が読んで直してから送り、自動では送信しない。
- 自然な文面の返信にしたい場合は、検索した FAQ を LLM に渡して書かせ、FAQ との照合で裏付けが取れない応答は使わない (この章の「資料を調べて答えるチャットボットを作る (複数ターンの RAG)」)。実際の問い合わせには氏名や注文番号、電話番号が含まれるので、外部の API に送る前にマスクする。電話番号やメールアドレスはこの章の「入力と出力に安全策を入れる (ガードレール)」のように正規表現で、氏名は固有表現抽出の章の「個人情報をマスクする (仮名化)」のように固有表現抽出で見つける。